# 03 · Episodic META-DDIE protocol (Tables 7.12 and 7.13)

Source: `metaddie_clean_run_CLAUDE` (cells 4–22).
1. META-DDIE graph: classes with ≥ 2 examples, 70/15/15 split over **common** classes (1–175) only.
2. Pretrain on common classes (hidden width 384, normalised LDAM).
3. 1-shot episodes, 5- and 10-way, on common-test / fewer / rare pools: nearest prototype and linear head → **Table 7.12**.
4. T3: ChemBERTa-only embeddings (no GNN).
5. T4: inference-time smoothing of the frozen embeddings → **Table 7.13**.

Known issue kept: episode seeds use `hash(split_name) % 7`, which Python randomises per process. To fix the episode draws across sessions, set `PYTHONHASHSEED` before the kernel starts (locally: `PYTHONHASHSEED=0 jupyter lab`).
On the test subgraph the fewer and rare pools usually have too few classes; those rows are skipped.

In [1]:
ON_COLAB  = False
REPO_DIR  = "/home/ppcrniai/Documents/ddi-linegraph"
DATA_ROOT = "/home/ppcrniai/Documents/ddi-linegraph/ddi_data"
WORK_ROOT = "/home/ppcrniai/Documents/ddi-linegraph/ddi_runs"
TEST_MODE = True
N_TEST_DRUGS = 800   # must match the test graph built in notebook 00

In [2]:
import os, sys
if ON_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    if not os.path.exists(os.path.join(REPO_DIR, "pyproject.toml")):
        raise SystemExit(f"Repo not found at {REPO_DIR}. Upload or clone it there first.")
    !bash {REPO_DIR}/setup/colab_setup.sh
# the editable install is only picked up by a fresh interpreter; make it importable now
if os.path.join(REPO_DIR, "src") not in sys.path:
    sys.path.insert(0, os.path.join(REPO_DIR, "src"))

In [3]:
import json, gc
import numpy as np
import torch
import dgl
import warnings
warnings.filterwarnings("ignore", message="Dataloader CPU affinity opt is not enabled")

from ddi.paths import get_paths
paths = get_paths(DATA_ROOT, WORK_ROOT, local_bin_dir=("/content" if ON_COLAB else None))
paths.make_output_dirs()
paths.show()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device, "| TEST_MODE:", TEST_MODE)

  data_root              /home/ppcrniai/Documents/ddi-linegraph/ddi_data
  work_root              /home/ppcrniai/Documents/ddi-linegraph/ddi_runs
  raw_data_dir           /home/ppcrniai/Documents/ddi-linegraph/ddi_data/data   (does not exist yet)
  drug_db                /home/ppcrniai/Documents/ddi-linegraph/ddi_data/data/Drug_META_DDIE.db   (does not exist yet)
  event_db               /home/ppcrniai/Documents/ddi-linegraph/ddi_data/data/event_META_DDIE_5foldLabel.db   (does not exist yet)
  graph_zip              /home/ppcrniai/Documents/ddi-linegraph/ddi_data/ddi_graph_and_linegraph.zip   (does not exist yet)
  graph_bin              /home/ppcrniai/Documents/ddi-linegraph/ddi_runs/ddi_graph_and_linegraph.bin   (does not exist yet)
  thesis_warm_ckpt_dir   /home/ppcrniai/Documents/ddi-linegraph/ddi_data/thesis_smart_fusion_whole_graph_checkpoints   (does not exist yet)
  thesis_hidden384_dir   /home/ppcrniai/Documents/ddi-linegraph/ddi_data/thesis_hidden384_experiment   (does not ex

/home/ppcrniai/Documents/ddi-linegraph/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
from ddi import graph as G, episodic as E
from ddi.splits import build_metaddie_graph, save_split, check_split

if TEST_MODE:
    drug_graph, line_graph = G.load_graph_pair(os.path.join(paths.test_graph_dir, f"test_pair_{N_TEST_DRUGS}.bin"))
else:
    G.ensure_graph_bin(paths.graph_zip, paths.graph_bin)
    drug_graph, line_graph = G.load_graph_pair(paths.graph_bin)
line_graph = G.attach_endpoint_features(drug_graph, line_graph)
print(f"line graph: {line_graph.num_nodes()} nodes, {line_graph.num_edges()} edges")

SEED = 42
E.set_seed(SEED)
g = build_metaddie_graph(line_graph, seed=SEED)
del line_graph; gc.collect()
RUN_TAG = f"test{N_TEST_DRUGS}" if TEST_MODE else "full"
split_path = os.path.join(paths.episodic_dir, f"split_{RUN_TAG}.pt")
check_split(g, split_path) if os.path.exists(split_path) else save_split(g, split_path, seed=SEED)

line graph: 81242 nodes, 8520237 edges
Classes with >= 2 samples: 128
Graph after min_class_samples filter: 81234 nodes (dropped 8)
Common nodes: 62304
Fewer nodes (176-204): 8
Rare nodes (205-227):  0
  Note: 11 class(es) too small to stratify val/test; moving their 11 sample(s) back to train.
Split: train=43623 | val=9340 | test=9341
saved split -> /home/ppcrniai/Documents/ddi-linegraph/ddi_runs/episodic/split_test800.pt


## Pretrain on common classes
`USE_THESIS_CHECKPOINT = True` loads `thesis_metaddie_clean_run/pretrain_common_only.pt` instead (full graph only).

In [5]:
USE_THESIS_CHECKPOINT = False
if USE_THESIS_CHECKPOINT:
    assert not TEST_MODE
    model = E.load_pretrained(g, os.path.join(paths.thesis_episodic_dir, "pretrain_common_only.pt"), device)
else:
    ckpt = os.path.join(paths.episodic_dir, f"pretrain_common_only_{RUN_TAG}.pt")
    model = E.train_common_only(g, device, ckpt, epochs=(2 if TEST_MODE else 100), batch_size=512,
                                fanouts=(15, 10), monitor="macro_f1", patience=20)

Train=43623, Val=9340, classes=175
Monitor: val_macro_f1, patience=20, epochs=2


[epoch 001] loss=9.8832 | train acc=0.3776 f1=0.0310 | val acc=0.3738 f1=0.0339 | best=-inf@0 no_improve=0/20
  saved new best to /home/ppcrniai/Documents/ddi-linegraph/ddi_runs/episodic/pretrain_common_only_test800.pt


[epoch 002] loss=4.6957 | train acc=0.5378 f1=0.0901 | val acc=0.5273 f1=0.0954 | best=0.0339@1 no_improve=0/20
  saved new best to /home/ppcrniai/Documents/ddi-linegraph/ddi_runs/episodic/pretrain_common_only_test800.pt
loaded best checkpoint: epoch 2, val_acc=0.5273, val_macro_f1=0.0954


## Table 7.12: episodic accuracy

In [6]:
EPISODES = 200 if TEST_MODE else 10_000
Z = E.compute_all_embeddings(model, g, device)
labels_all_dev, pools = E.make_pools(g, device)
for name, pool in pools.items():
    for N in (5, 10):
        try:
            print(f"  {name:6s} N={N:<3d}: {len(E.eligible(pool, N, 1))} eligible classes")
        except ValueError as err:
            print(f"  {name:6s} N={N:<3d}: INSUFFICIENT ({err})")
results = E.run_episodic_suite(Z, labels_all_dev, pools, EPISODES, SEED)
json.dump(E.results_to_json(results), open(os.path.join(paths.episodic_dir, f"episodic_results_{RUN_TAG}.json"), "w"), indent=2)
print("META-DDIE paper:", E.METADDIE_PAPER)

  common N=5  : 96 eligible classes
  common N=10 : 96 eligible classes
  fewer  N=5  : INSUFFICIENT (Only 2 eligible labels; need 5.)
  fewer  N=10 : INSUFFICIENT (Only 2 eligible labels; need 10.)
  rare   N=5  : INSUFFICIENT (Only 0 eligible labels; need 5.)
  rare   N=10 : INSUFFICIENT (Only 0 eligible labels; need 10.)


common N=5  | proto acc=0.5970 f1=0.5818 | linhead acc=0.5820 f1=0.5572


common N=10 | proto acc=0.4765 f1=0.4643 | linhead acc=0.4885 f1=0.4530
[skip] fewer N=5: Only 2 eligible labels; need 5.
[skip] fewer N=10: Only 2 eligible labels; need 10.
[skip] rare N=5: Only 0 eligible labels; need 5.
[skip] rare N=10: Only 0 eligible labels; need 10.
META-DDIE paper: {5: {'META-DDIE': {'common': 0.8379, 'fewer': 0.8263, 'rare': 0.7123}, 'Transf. DeepDDI': {'common': 0.7727, 'fewer': 0.7464, 'rare': 0.6892}}, 10: {'META-DDIE': {'common': 0.7743, 'fewer': 0.7521, 'rare': 0.6437}, 'Transf. DeepDDI': {'common': 0.7603, 'fewer': 0.7136, 'rare': 0.6163}}}


## T3: ChemBERTa-only baseline

In [7]:
t3 = {}
for mode in ["mean", "hadamard", "set_untrained"]:
    print(f"\n=== ChemBERTa-only, fusion = {mode}")
    Zc = E.build_chemberta_only_embeddings(g, device, fusion_mode=mode, seed=SEED)
    r = E.run_episodic_suite(Zc, labels_all_dev, pools, EPISODES, SEED, seed_salt=mode)
    t3.update({(mode,) + k: v for k, v in r.items()})
    del Zc; torch.cuda.empty_cache()
json.dump(E.results_to_json(t3), open(os.path.join(paths.episodic_dir, f"t3_chemberta_only_{RUN_TAG}.json"), "w"), indent=2)


=== ChemBERTa-only, fusion = mean


common N=5  | proto acc=0.4950 f1=0.4846 | linhead acc=0.4990 f1=0.4668


common N=10 | proto acc=0.3895 f1=0.3652 | linhead acc=0.4030 f1=0.3745
[skip] fewer N=5: Only 2 eligible labels; need 5.
[skip] fewer N=10: Only 2 eligible labels; need 10.
[skip] rare N=5: Only 0 eligible labels; need 5.
[skip] rare N=10: Only 0 eligible labels; need 10.

=== ChemBERTa-only, fusion = hadamard


common N=5  | proto acc=0.2700 f1=0.2542 | linhead acc=0.3080 f1=0.3076


common N=10 | proto acc=0.1945 f1=0.1876 | linhead acc=0.1995 f1=0.1945
[skip] fewer N=5: Only 2 eligible labels; need 5.
[skip] fewer N=10: Only 2 eligible labels; need 10.
[skip] rare N=5: Only 0 eligible labels; need 5.
[skip] rare N=10: Only 0 eligible labels; need 10.

=== ChemBERTa-only, fusion = set_untrained


common N=5  | proto acc=0.4680 f1=0.4520 | linhead acc=0.5100 f1=0.4854


common N=10 | proto acc=0.3560 f1=0.3421 | linhead acc=0.3615 f1=0.3318
[skip] fewer N=5: Only 2 eligible labels; need 5.
[skip] fewer N=10: Only 2 eligible labels; need 10.
[skip] rare N=5: Only 0 eligible labels; need 5.
[skip] rare N=10: Only 0 eligible labels; need 10.


## T4: inference-time smoothing (Table 7.13)

In [8]:
E.set_seed(SEED)
train_nids = g.ndata["train_mask"].nonzero(as_tuple=False).squeeze().to(device)
knn_idx, knn_sim = E.build_knn_to_train(Z, train_nids, k=5)
apply_mask = (((labels_all_dev >= E.FEWER_MIN) & (labels_all_dev <= E.FEWER_MAX))
              | ((labels_all_dev >= E.RARE_MIN) & (labels_all_dev <= E.RARE_MAX)))
print(f"smoothing applies to {int(apply_mask.sum())} nodes (fewer + rare)")
t4 = {}
for cfg, mode, params in E.T4_FUSION_CONFIGS:
    print(f"\n=== smoothing {cfg}")
    Zs = E.compute_smoothed_Z(Z, knn_idx, knn_sim, apply_mask, fusion_mode=mode, fusion_params=params)
    r = E.run_episodic_suite(Zs, labels_all_dev, pools, EPISODES, SEED, seed_salt=cfg)
    t4.update({(cfg,) + k: v for k, v in r.items()})
    del Zs; torch.cuda.empty_cache()
json.dump(E.results_to_json(t4), open(os.path.join(paths.episodic_dir, f"t4_smoothing_{RUN_TAG}.json"), "w"), indent=2)

kNN: querying 81234 nodes against 43623 training neighbors, k=5


smoothing applies to 8 nodes (fewer + rare)

=== smoothing alpha_a0.7


common N=5  | proto acc=0.5970 f1=0.5803 | linhead acc=0.6210 f1=0.6087


common N=10 | proto acc=0.4765 f1=0.4511 | linhead acc=0.4955 f1=0.4650
[skip] fewer N=5: Only 2 eligible labels; need 5.
[skip] fewer N=10: Only 2 eligible labels; need 10.
[skip] rare N=5: Only 0 eligible labels; need 5.
[skip] rare N=10: Only 0 eligible labels; need 10.

=== smoothing alpha_a0.5


common N=5  | proto acc=0.5970 f1=0.5818 | linhead acc=0.5820 f1=0.5548


common N=10 | proto acc=0.4765 f1=0.4643 | linhead acc=0.4965 f1=0.4618
[skip] fewer N=5: Only 2 eligible labels; need 5.
[skip] fewer N=10: Only 2 eligible labels; need 10.
[skip] rare N=5: Only 0 eligible labels; need 5.
[skip] rare N=10: Only 0 eligible labels; need 10.

=== smoothing alpha_a0.9


common N=5  | proto acc=0.6020 f1=0.5915 | linhead acc=0.6010 f1=0.5802


common N=10 | proto acc=0.4750 f1=0.4576 | linhead acc=0.4815 f1=0.4611
[skip] fewer N=5: Only 2 eligible labels; need 5.
[skip] fewer N=10: Only 2 eligible labels; need 10.
[skip] rare N=5: Only 0 eligible labels; need 5.
[skip] rare N=10: Only 0 eligible labels; need 10.

=== smoothing replace


common N=5  | proto acc=0.5970 f1=0.5803 | linhead acc=0.6200 f1=0.6080


common N=10 | proto acc=0.4765 f1=0.4511 | linhead acc=0.4925 f1=0.4618
[skip] fewer N=5: Only 2 eligible labels; need 5.
[skip] fewer N=10: Only 2 eligible labels; need 10.
[skip] rare N=5: Only 0 eligible labels; need 5.
[skip] rare N=10: Only 0 eligible labels; need 10.
